<div dir="rtl">

# תבניות בטקסט: ביטויים רגולריים

</div>

<div dir="rtl">

בשיעור הזה נלמד מהו [ביטוי רגולרי](https://dvirross.com/he/courses/intro-data-science/glossary#regular-expression) (regular expression), איך המודול re של Python מוצא את הקטעים בטקסט שמתאימים לתבנית, איך תווים, חזרות וקבוצות מתארים את התבנית, ומתי תבנית היא הכלי הנכון ומתי Beautiful Soup.

## איפה מספר העמודים?

מחברים שמעלים מאמר ל־[arXiv](https://arxiv.org) יכולים להוסיף שורת הערות, שנכתבת כטקסט חופשי: `7 pages, 5 figures. Accepted to ASP-DAC 2027`. אנחנו רוצים את מספר העמודים של כל מאמר שאומר אותו, למשל כדי לראות כמה ארוכים מאמרים בארכיטקטורת חומרה. Beautiful Soup מביאה אותנו לאלמנט שמחזיק את ההערה, כמו ב[מאלמנט אחד לכל הרשומות: find_all, מספרים וערכים חסרים](https://dvirross.com/he/courses/intro-data-science/data-collection-find-all). למספר שבתוך הטקסט אין אלמנט משלו: זה קטע טקסט שיש לו צורה, ספרות, רווח והמילה `pages`.

אפשר לחתוך טקסט עם `split` ו־`replace`, וזה עובד כל עוד המספר נמצא במקום ידוע. **ביטוי רגולרי** מתאר את הצורה ולא את המקום: התוכנית מבקשת ספרות ואחריהן המילה, בכל מקום שהן נמצאות בטקסט. מי שאוסף טקסט מהאינטרנט פוגש את הצורך הזה: מחירים, תאריכים, מספרי טלפון, מזהים וגדלים נמצאים בתוך משפטים.

</div>

<div dir="rtl">

## הכנה: הספריות ופונקציית הבדיקה

אין צורך בחשבון ואין צורך במפתח. התא הבא מייבא את `requests` ואת `beautifulsoup4` (ב־Colab יש את שתיהן; התא מתקין אותן אם בסביבה אין). הדפים שהקוד קורא הם עותקים שמורים של דפי arXiv, ששמורים באתר הציבורי של הקורס, ולכן המספרים זהים לכולם. המחברת הזו לא שולחת בקשות ל־arXiv עצמו.

</div>

In [1]:
try:
    import bs4, requests
except ImportError:
    !pip install -q requests beautifulsoup4
    import bs4, requests
print('requests', requests.__version__, '| beautifulsoup4', bs4.__version__)

requests 2.34.2 | beautifulsoup4 4.15.0


<div dir="rtl">

התא הבא מגדיר את הפונקציה `check(task_number)`. אחרי שמריצים את התא של משימה, מריצים את התא שמתחתיו: `check` מריצה שוב את התא שלכם, אוספת את מה שהוא מדפיס ומשווה אותו לפלט הצפוי. היא אומרת אם הפלט נכון, ואם לא באיזו שורה הוא שונה; היא לא מראה את הפתרון.

</div>

In [2]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '1': [
        '79eb0b37c90d'
    ],
    '2': [
        '4fc82b26aecb'
    ],
    '3': [
        '9b7d87bc5316'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('אין משימה כזו: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('התא נכשל בשגיאה:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('לא הודפס כלום. הדפיסו את התשובה בתא המשימה, הריצו אותו, והריצו את `check` מיד אחריו.')
    elif len(got) != len(want):
        print('התא הדפיס {got} שורות והמשימה מצפה ל־{want}. קראו שוב את המשימה ואת הרמז.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('שורה {k} שונה מהפלט הצפוי. קראו שוב את המשימה ואת הרמז.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: הפלט כצפוי.')


<div dir="rtl">

התיבה הראשונה מבקשת את העותק השמור של הרשימה של אוקטובר 2026 ובונה את ה־`soup`, כמו ב[מדף אינטרנט לנתונים: Beautiful Soup](https://dvirross.com/he/courses/intro-data-science/data-collection-beautifulsoup), כדי שהדף הזה יעמוד בפני עצמו. אחר כך היא שמה ברשימה `comments` את הטקסט של ההערה של כל מאמר. מ־25 המאמרים, ל־12 יש שורת הערות.

</div>

In [3]:
import requests
from bs4 import BeautifulSoup

base_url = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/'
)
list_url = base_url + 'arxiv-cs-ar-2026-10.htm'
page_text = requests.get(list_url).text
soup = BeautifulSoup(page_text, 'html.parser')

boxes = soup.find_all('div', class_='list-comments')
comments = [
    box.text.replace('Comments:', '').strip()
    for box in boxes
]
print(len(comments))
print(comments[1])

12
7 pages, 5 figures. Accepted to ASP-DAC 2027


<div dir="rtl">

חיתוך לפי מקום עובד בשביל העמודים, כי המספר הוא הדבר הראשון בשורה. נבקש מספר אחר: כמה טבלאות יש במאמר. ההערה אומרת זאת אחרי העמודים והאיורים, ורק חלק מההערות אומרות זאת בכלל. השורה הראשונה שלמטה לוקחת את הקטע השלישי של הערה שנחתכה בפסיקים, ואת המילה הראשונה שלו; השורה השנייה חותכת הערה אחרת באותה דרך:

</div>

In [4]:
print(comments[2].split(', ')[2].split(' ')[0])
print(comments[5].split(', ')[2])

3
Hong Kong


<div dir="rtl">

השורה הראשונה נותנת `3`, מספר הטבלאות של המאמר. השורה השנייה רצה בלי שגיאה ועונה `Hong Kong`: ההערה הזאת לא מזכירה טבלאות, והקטע השלישי בה הוא מקום. חיתוך לפי מקום אינו יודע מה חתך, ואינו נעצר כשהשורה כתובה בדרך אחרת. תבנית יכולה לבקש מספר ואחריו המילה `tables` בכל מקום בשורה, וזה מה שהשיעור מלמד.

## `re.search`: הקטע הראשון שמתאים

המודול `re` מגיע עם Python, ו־`import re` מייבא אותו. **[תבנית](https://dvirross.com/he/courses/intro-data-science/glossary#pattern)** (pattern) היא מחרוזת שמתארת את הטקסט שנחפש. נכתוב אותה בתור **[מחרוזת גולמית](https://dvirross.com/he/courses/intro-data-science/glossary#raw-string)** (raw string): מחרוזת עם האות `r` לפני הגרש הפותח, כמו `r'pages'`. במחרוזת גולמית Python משאירה כל לוכסן הפוך כמו שהוא, והתבניות בהמשך משתמשות בהרבה לוכסנים הפוכים, ולכן כל תבנית בשיעור הזה היא מחרוזת גולמית.

הפונקציה `re.search(pattern, text)` מחפשת בטקסט את המקום הראשון שהתבנית מתאימה בו. היא מחזירה **[אובייקט התאמה](https://dvirross.com/he/courses/intro-data-science/glossary#match)** (match object) שמתאר את המקום, או `None` כשהתבנית אינה מתאימה בשום מקום. השיטה `group()` של אובייקט התאמה מחזירה את קטע הטקסט שהתאים.

</div>

In [5]:
import re

text = comments[1]
match = re.search(r'pages', text)
print(match)
print(match.group())
print(re.search(r'tables', text))

<re.Match object; span=(2, 7), match='pages'>
pages
None


<div dir="rtl">

אובייקט ההתאמה אומר שהקטע מתחיל במיקום 2 ונגמר לפני מיקום 7. כמו עם `find`, תוצאה שיכולה להיות `None` נבדקת עם `is None` לפני שנקרא עליה ל־`group()`.

## תווים ומחלקות תווים

נקרא תבנית משמאל לימין, וכל קטע שלה מתאים לתו אחד של הטקסט. אות מתאימה לעצמה. **מחלקת תווים** (character class) היא קטע שמתאים לכל תו אחד מסוג מסוים:

| הקטע | מתאים לתו אחד שהוא | דוגמה בהערה |
|---|---|---|
| `\d` | ספרה, 0 עד 9 | `7` |
| `[0-9]` | ספרה: כל תו בטווח שבסוגריים המרובעים | `7` |
| `[pf]` | אחד מהתווים שבסוגריים המרובעים | ה־`p` של `pages` או ה־`f` של `figures` |
| `\w` | אות, ספרה או קו תחתון | `p`, `5` |
| `\s` | רווח לבן: רווח, טאב או מעבר שורה | הרווח אחרי פסיק |
| `.` | כל תו חוץ ממעבר שורה | כל תו |

לתווים מסוימים יש משמעות משלהם בתבנית, כמו `.`, `[` ו־`(`. כדי להתאים לתו עצמו, נכתוב לפניו לוכסן הפוך: `\.` מתאים לנקודה ו־`\[` לסוגר מרובע.

</div>

In [6]:
print(re.search(r'\d pages', comments[1]))
print(re.search(r'[0-9] pages', comments[1]))
print(re.search(r'figures.\s\w', comments[1]))

<re.Match object; span=(0, 7), match='7 pages'>
<re.Match object; span=(0, 7), match='7 pages'>
<re.Match object; span=(11, 21), match='figures. A'>


<div dir="rtl">

**נסו בעצמכם.** ההערה של מאמר אחר מתחילה ב־`15 pages.` לאיזה קטע של ההערה מתאימה התבנית `\d pages`? נבאו את הקטע, ואחר כך הריצו את התיבה.

</div>

In [7]:
print(re.search(r'\d pages', comments[9]).group())

5 pages


<div dir="rtl">

## כמתים: כמה פעמים

**כמת** (quantifier) שנכתב אחרי קטע אומר כמה פעמים הקטע חוזר. הוא שייך לקטע שנמצא ממש לפניו.

| הכמת | משמעות | דוגמה |
|---|---|---|
| `+` | פעם אחת או יותר | `\d+` מתאים ל־`7` ול־`15` |
| `*` | אפס פעמים או יותר | `\d*` מתאים ל־`15`, וגם לכלום |
| `?` | אפס פעמים או פעם אחת: הקטע אינו חובה | `pages?` מתאים ל־`page` ול־`pages` |
| `{n}` | בדיוק n פעמים | `\d{4}` מתאים ל־`2026` |

הפונקציה `re.findall(pattern, text)` מחזירה רשימה של כל קטעי הטקסט שמתאימים, משמאל לימין. כל קטע מתחיל אחרי הסוף של הקטע הקודם, ולכן שני קטעים אינם חולקים אותם תווים; למשל `\d\d` על `2610` מוצאת `26` ו־`10`, ולא גם `61`. קטע שחוזר בטקסט מופיע ברשימה בכל פעם שהוא נמצא, כמו `3` בשורה השנייה של התיבה שלמטה. כל פריט הוא מחרוזת. בשורה השנייה של התיבה שלמטה התבנית מוצאת כל רצף ספרות בהערה, ובשורה השלישית היא מוצאת את המזהה של כל מאמר ברשימה: ארבע ספרות, נקודה וחמש ספרות. המשתנה `soup.text` הוא כל הטקסט של הדף, בלי התגיות שלו.

</div>

In [8]:
print(re.search(r'\d+ pages', comments[9]).group())
print(re.findall(r'\d+', comments[2]))
ids = re.findall(r'\d{4}\.\d{5}', soup.text)
print(len(ids), ids[:3])

15 pages
['4', '3', '3', '2026']
25 ['2610.00106', '2610.00207', '2610.00311']


<div dir="rtl">

## קבוצות: לוקחים חלק מההתאמה

התבנית `\d+ pages` מתאימה לטקסט `15 pages`, אבל אנחנו רוצים רק את המספר, ו־`group()` מחזירה את הקטע כולו. סוגריים עגולים מסמנים את החלק שרוצים להוציא: `(\d+) pages` מתאימה לאותו טקסט, כי הסוגריים אינם משנים מה מתאים, והקטע שבתוכם נשמר גם בנפרד. זו **[קבוצת לכידה](https://dvirross.com/he/courses/intro-data-science/glossary#capturing-group)** (capturing group). השיטה `group(1)` מחזירה את הקטע של זוג הסוגריים הראשון, ו־`group(2)` את הקטע של השני, כשסופרים לפי הסוגר הפותח משמאל לימין; `group()` ממשיכה להחזיר את ההתאמה כולה. קבוצה מחזירה טקסט, ולכן `int` ממירה אותו למספר.

</div>

In [9]:
match = re.search(r'(\d+) pages, (\d+) figures', comments[1])
print(match.group())
print(match.group(1), match.group(2))
print(int(match.group(1)) + 1)

7 pages, 5 figures
7 5
8


<div dir="rtl">

כשבתבנית יש קבוצות, `re.findall` מחזירה את הקטעים של הקבוצות במקום את הקטע כולו: רשימה של מחרוזות לקבוצה אחת, ורשימה של טאפלים כשיש כמה קבוצות.

</div>

In [10]:
print(re.findall(r'(\d+) pages', comments[2]))
print(re.findall(r'(\d+) pages, (\d+) figures', comments[2]))

['4']
[('4', '3')]


<div dir="rtl">

### תרגול

**נסו בעצמכם.** ברשימה `comments` מופיעה ההערה של כל מאמר שיש לו הערה. הדפיסו את הרשימה של מספרי העמודים, כמספרים שלמים, של המאמרים שההערה שלהם אומרת כמה עמודים יש להם, לפי הסדר של `comments`.

</div>

In [11]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
[<number>, <number>, <number>, <number>, <number>, <number>]
```

</div>

In [12]:
check(1)

<div dir="rtl">

> **רמז**
>
> עברו על ההערות בלולאה. בכל אחת, `re.search` עם קבוצה סביב הספרות מחזירה אובייקט התאמה או `None`; בדקו אותו לפני שאתם קוראים ל־`group(1)`, והמירו את הקטע עם `int`.

</div>

<div dir="rtl">

## הגודל והתאריך של הגשה

בדף של מאמר יש קטע שנקרא היסטוריית ההגשות. הטקסט שלו מחזיק, בשורה אחת, את התאריך ואת הגודל של הגרסה הראשונה, `[v1]`, של המאמר: `[v1] Wed, 9 Sep 2026 02:40:29 UTC (3,330 KB)`. מספר בתוך טקסט אפשר לחתוך עם `split`, `replace` ו־`strip`, כמו ב[מאלמנט אחד לכל הרשומות: find_all, מספרים וערכים חסרים](https://dvirross.com/he/courses/intro-data-science/data-collection-find-all); תבנית מתארת ישירות את המספר ואת היחידה שלו.

הפונקציה `history_of` מבקשת את הדף השמור של מאמר לפי המזהה שלו ומחזירה את הטקסט של היסטוריית ההגשות שלו. דפים שמורים יש לשמונת המאמרים הראשונים ברשימה.

</div>

In [13]:
def history_of(paper_id):
    address = base_url + 'arxiv-abs-' + paper_id + '.htm'
    page = BeautifulSoup(requests.get(address).text, 'html.parser')
    return page.find('div', class_='submission-history').text

history = history_of(ids[0])
line = re.search(r'\[v1\]\s+(.*)', history).group(1)
print(line)

Wed, 9 Sep 2026 02:40:29 UTC (3,330 KB)


<div dir="rtl">

התבנית `\[v1\]\s+(.*)` מתאימה לסוגר המרובע, ל־`v1` ולסוגר, אחר כך ל־`\s+`, מעבר השורה והרווחים שאחריה בדף, ולקבוצה `(.*)`: כל מספר של תווים עד סוף השורה, כי הנקודה אינה מתאימה למעבר שורה.

</div>

<div dir="rtl">

**למה Beautiful Soup וגם תבנית, ולא אחת מהן לבדה?**

</div>

<div dir="rtl">

**תבנית לבדה** צריכה לרוץ על ה־HTML הגולמי, שבו אחרי `[v1]` בא תג סוגר: `<strong>[v1]</strong>`. התבנית `\[v1\]\s+(.*)` לא מוצאת שם כלום, ותבנית שכוללת את התגיות, `\[v1\]</strong>\s+(.*?)<br`, עובדת רק כל עוד הדף כותב את התגיות שלו כך: עם `<b>` במקום `<strong>`, או עם `<br>` במקום `<br/>`, היא לא מוצאת כלום ואינה מדווחת על שגיאה.

**Beautiful Soup לבדה** מוצאת את האלמנט ונותנת את הטקסט שלו, אבל את השורה צריך אחר כך לחתוך ממנו לפי מקום, עם `split`, ואת הגודל עם `replace`. זה עובד כל עוד הדף שומר על הסדר של החלקים, ונשבר כשהסדר משתנה; התבנית `([0-9,]+) KB` מתאימה לגודל בכל מקום שבו הוא כתוב.

כל כלי עושה את מה שהוא מכיר: Beautiful Soup מכירה את המבנה של הדף, ולכן היא מוצאת את האלמנט ונותנת את הטקסט שלו בלי תגיות; התבנית מכירה את הצורה של טקסט, ולכן היא לוקחת את הגודל ואת התאריך בכל מקום שבו הם עומדים בשורה. בשמונת הדפים השמורים כלי אחד היה נותן גם הוא את התשובה הנכונה; שני הכלים יחד מונעים מהקוד להיות תלוי בתגיות ובמקומות.

</div>

<div dir="rtl">

בשורה, הגודל הוא ספרות ופסיקים ואחריהם `KB`, והתאריך הוא יום, חודש ושנה. מחלקת התווים `[0-9,]` מתאימה לספרה או לפסיק (לפסיק אין משמעות מיוחדת בתוך הסוגריים המרובעים), ולכן `[0-9,]+` מתאים ל־`3,330`. הקטע `\d\d?` הוא ספרה וספרה שנייה שאינה חובה: ליום בחודש יש ספרה אחת או שתיים. הטקסט `3,330` הוא עדיין טקסט, ו־`replace` מסירה את הפסיק לפני ש־`int` ממירה אותו.

</div>

In [14]:
size = re.search(r'([0-9,]+) KB', line).group(1)
print(size)
print(int(size.replace(',', '')))
print(re.findall(r'(\d\d?) (\w+) (\d{4})', line))

3,330
3330
[('9', 'Sep', '2026')]


<div dir="rtl">

## הנקודה וההתאמה הקצרה ביותר

הנושאים של מאמר כתובים בשורה כמו `Hardware Architecture (cs.AR); Artificial Intelligence (cs.AI)`. הקוד של כל נושא נמצא בסוגריים עגולים. בתבנית, `\(` ו־`\)` מתאימים לסוגריים העגולים עצמם, וזוג הסוגריים בלי לוכסן הפוך הוא הקבוצה ששומרת את מה שביניהם.

התיבה שלמטה מגדירה את הפונקציה `subjects_of`, שמחזירה את שורת הנושאים של רשומה אחת, ובונה את הרשימה `subjects` של 25 המאמרים. היא רק מכינה את הנתונים עם Beautiful Soup, כמו בשיעורים הקודמים; העבודה עם תבניות מתחילה בתיבה שאחריה. למאמר השני יש שלושה נושאים.

</div>

In [15]:
def subjects_of(entry):
    box = entry.find('div', class_='list-subjects')
    return box.text.replace('Subjects:', '').strip()

subjects = [subjects_of(e) for e in soup.find_all('dd')]
print(subjects[1])

Hardware Architecture (cs.AR); Artificial Intelligence (cs.AI); Image and Video Processing (eess.IV)


<div dir="rtl">

הקטע `.*` מתאים לכל מספר של תווים, והוא לוקח כמה שהוא יכול כל עוד התבנית כולה עדיין מתאימה: הוא **חמדן** (greedy). ב־`\((.*)\)` הקבוצה מתחילה אחרי הסוגר העגול הפותח הראשון ורצה עד הסוגר העגול האחרון בשורה. סימן `?` אחרי כמת גורם לו לקחת כמה שפחות תווים, **ההתאמה הקצרה ביותר**: `.*?` נעצר בסוגר העגול הראשון שמאפשר לתבנית להתאים.

</div>

In [16]:
print(re.findall(r'\((.*)\)', subjects[1]))
print(re.findall(r'\((.*?)\)', subjects[1]))

['cs.AR); Artificial Intelligence (cs.AI); Image and Video Processing (eess.IV']
['cs.AR', 'cs.AI', 'eess.IV']


<div dir="rtl">

אותו תו, `?`, משמעותו כפולה: אחרי קטע הוא מאפשר לו לא להופיע, ואחרי כמת הוא גורם לכמת לקחת כמה שפחות.

### תרגול

**נסו בעצמכם.** ברשימה `subjects` מופיעה שורת הנושאים של כל אחד מ־25 המאמרים. הקוד של נושא נמצא בסוגריים עגולים. הדפיסו את מספר המאמרים שיש להם יותר מנושא אחד.

</div>

In [17]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
<number>
```

</div>

In [18]:
check(2)

<div dir="rtl">

> **רמז**
>
> `re.findall` עם קבוצה סביב הקוד מחזירה את הקודים של מאמר אחד כרשימה; למאמר יש יותר מנושא אחד כשברשימה יותר מפריט אחד. בדקו שהקבוצה נעצרת בסוגר העגול הראשון.

</div>

<div dir="rtl">

## בדיקת תבנית בכלי

תבנית שנראית נכונה יכולה להתאים למשהו אחר, ולכן נבדוק אותה על טקסט אמיתי לפני שנריץ אותה על אלף דפים. [מעבדת הביטויים הרגולריים](https://dvirross.com/he/courses/intro-data-science/data-collection-regex-practice) של הקורס היא כלי לבדיקה כזו: הטקסט נכנס לתיבה אחת והתבנית לתיבה אחרת, כל קטע שמתאים מודגש, ורשימה מסבירה כל חלק של התבנית במילים פשוטות. במעבדה נכתוב תבנית בלי מרכאות ובלי האות `r`.

כדי לנסות, פתחו את המעבדה, השאירו את הערות המאמרים כטקסט, וכתבו `(\d+) pages, (\d+) figures` כתבנית: המעבדה מסמנת את ההתאמה כולה ואת שתי הקבוצות של ההערות שמתאימות. אחר כך שנו בטקסט את `7 pages` ל־`1 page` וראו מדוע התבנית כבר אינה מתאימה לשורה הזאת.

## מתי ביטוי רגולרי אינו הכלי

ביטוי רגולרי קורא טקסט ואינו יודע דבר על דף. Beautiful Soup בונה את עץ האלמנטים ומוצאת אלמנט לפי התגית שלו, ה־`class` שלו או התכונה שלו. תבנית שמופעלת על טקסט ה־HTML הגולמי של דף תלויה בפרטי הכתיבה של התגיות, והיא נכשלת בשקט: היא מחזירה רשימה ריקה ולא שגיאה. התיבה סופרת, עם תבנית ועם Beautiful Soup, את המזהים ואת אלמנטי ההערות ברשימה:

</div>

In [19]:
print(len(re.findall(r'\d{4}\.\d{5}', page_text)))
print(len(re.findall(r'\d{4}\.\d{5}', soup.text)))
print(len(re.findall(r'<div class="list-comments">', page_text)))
print(len(soup.find_all('div', class_='list-comments')))

300
25
0
12


<div dir="rtl">

ה־HTML חוזר על כל מזהה 12 פעמים, בכתובות ובתכונות של הקישורים שלו (25 כפול 12 הוא 300), והטקסט של הדף נותן 25. התבנית לא מוצאת כלום משתי סיבות: הדף כותב את התכונה עם גרשים בודדים, `class='list-comments mathjax'`, ובתבנית יש גרשיים כפולים; ול־`class` יש מילה שנייה, `mathjax`, ולכן `<div class="list-comments">` לא היה מתאים גם עם הגרשיים הנכונים. `class_='list-comments'` לא תלוי באף אחד מהם, ומוצא את 12 האלמנטים.

חלוקת העבודה היא אפוא זו: Beautiful Soup מוצאת את האלמנט שמחזיק את הטקסט, ותבנית לוקחת את המספר, התאריך או הקוד מתוך הטקסט של האלמנט הזה. תבנית גם מתארת צורה ולא משמעות: בתקציר של המאמר EdgeDAE, `\d+\.\d+` מתאימה לאחוזים `52.5` ו־`38.7`, וגם ל־`2.10`, שהוא יחס של תפוקה. נסתכל על הקטעים שמתאימים, נספור אותם, ונבדוק את התבנית על טקסטים שאמורים לא להתאים.

## נקודות שכדאי לזכור

- Beautiful Soup מוצאת אלמנט לפי המבנה של הדף; ביטוי רגולרי מוצא תבנית בתוך קטע טקסט. נקח את ה־`text` של האלמנט, ואז נחפש בו עם תבנית.
- תבנית היא מחרוזת גולמית, `r'...'`. `re.search` מחזירה אובייקט התאמה, ש־`group()` שלו הוא הקטע שהתאים, או `None`; `re.findall` מחזירה רשימה של כל הקטעים שמתאימים, כטקסט.
- `\d`, `\w`, `\s`, `[0-9]`, `[abc]` ו־`.` מתאימים לתו אחד; לוכסן הפוך לפני `.`, `[` או `(` מתאים לתו עצמו.
- `+`, `*`, `?` ו־`{n}` אומרים כמה פעמים קטע חוזר; סוגריים עגולים שומרים חלק מההתאמה, וכשיש קבוצות `re.findall` מחזירה את הקבוצות.
- `.*` לוקח כמה שיותר ו־`.*?` כמה שפחות; קבוצה מהצורה `\((.*?)\)` נעצרת בסוגר העגול הראשון.
- קטע שנלקח מטקסט הוא טקסט: `int` ממירה ספרות, אחרי ש־`replace` מסירה פסיק. תבנית שמתאימה לדוגמאות יכולה להתאים גם לטקסט אחר, ולכן נבדוק אותה ונספור מה שהיא מוצאת.

</div>

<div dir="rtl">

> **ללימוד עצמי, חשיבות 1.** קטע ללימוד עצמי: הוא אינו נלמד בהרצאה, ונקרא אותו בעצמנו. קומפילציה של תבנית פעם אחת ושימוש בה פעמים רבות. הקטע מסייע בנושא: מתן שם לתבנית ושימוש חוזר בה בפונקציה.

</div>

<div dir="rtl">

## `re.compile`: תבנית עם שם

תבנית שתוכנית משתמשת בה פעמים רבות אפשר לבנות פעם אחת. הפונקציה `re.compile(pattern)` מחזירה תבנית מקומפלת, אובייקט שמחזיק את התבנית ויש לו השיטות `search` ו־`findall` משלו, כך ש־`compiled.search(text)` עושה מה ש־`re.search(pattern, text)` עושה. הפונקציה `pages_of` שלמטה מחזירה את מספר העמודים של הערה, או `None`, עם תבנית שנבנית פעם אחת.

</div>

In [20]:
pages_pattern = re.compile(r'(\d+) pages')

def pages_of(text):
    match = pages_pattern.search(text)
    if match is None:
        return None
    return int(match.group(1))

all_pages = [pages_of(text) for text in comments]
print(all_pages)
known = [pages for pages in all_pages if pages is not None]
print(len(known), max(known), sum(known) / len(known))

[None, 7, 4, 4, None, None, 8, None, 10, 15, None, None]
6 15 8.0


<div dir="rtl">

ששת המאמרים שמציינים כמה עמודים יש להם הם אותם שישה שמצאה הפונקציה שחותכת את הטקסט עם `split`, ב[מאלמנט אחד לכל הרשומות: find_all, מספרים וערכים חסרים](https://dvirross.com/he/courses/intro-data-science/data-collection-find-all); כאן התבנית מוצאת את המספר בכל מקום שהוא נמצא בהערה.

</div>

<div dir="rtl">

> **ללימוד עצמי, חשיבות 2.** קטע ללימוד עצמי: הוא אינו נלמד בהרצאה, ונקרא אותו בעצמנו. משימת תרגול שקוראת כמה דפים ולוקחת גודל מכל אחד. הקטע מסייע בנושא: ביטוי רגולרי שמופעל על הרבה דפים.

</div>

<div dir="rtl">

### תרגול: כמה דפים

**נסו בעצמכם.** הפונקציה `history_of` מחזירה את הטקסט של היסטוריית ההגשות של מאמר לפי המזהה שלו, והרשימה `ids` מחזיקה את המזהים של 25 המאמרים; דפים שמורים יש לשמונת הראשונים. הדפיסו את הגודל הגדול ביותר, כמספר שלם של KB, בין שמונת המאמרים הראשונים.

</div>

In [21]:
# Your turn

<div dir="rtl">

**פלט צפוי (המספרים מוסתרים)**

```text
<number>
```

</div>

In [22]:
check(3)

<div dir="rtl">

> **רמז**
>
> עברו על `ids[:8]`, ולכל מזהה קחו את הטקסט של הגודל עם קבוצה סביב הספרות והפסיק, הסירו את הפסיק והמירו עם `int`. השוו מספרים, לא טקסטים.

</div>